# 01: Metadata Inspection

This notebook is dedicated to the initial analysis of the `.vrs` file and its linked metadata. The primary goal is to validate the data container structure, identify all active sensor streams, and verify the consistency of the recorded samples against the nominal parameters of the Project Aria Gen 1 device.

As a sample, we will use `recording_test.vrs` and its associated `recording_test.json` metadata, located in the local `data/raw/recording_test/` directory.

### In this notebook, we will:

1. Initialize the Data Provider: Establish a connection with the `.vrs` file using the projectaria_tools library.

2. Stream Discovery: List and categorize all available Stream.

3. JSON Metadata Parsing: parse recording JSON and extract main metadata fields.

4. Temporal Consistency Checks: for each stream and across streams

5. Nominal vs Measured Rate Validation: Compute measured rate from timestamps and compare against nominal rate.

6. Basic Data Integrity Checks: Attempt decode/read of a small sample window per stream.

7. Exploratory Visualization: Plot stream activity on a shared timeline and Plot IMU accel/gyro segments

8. Export Reports: Save summary artifacts under:
    - `data/processed/metadata_summary.json`
    - `data/processed/stream_quality_report.csv`


### Definition of Done:

When this notebook is complete we should be able to validate that:

- VRS and JSON are parsed successfully  
- All streams are listed and categorized  
- Sample count and time bounds are computed per stream  
- Duplicates/out-of-order/missing-gap diagnostics are available  
- Nominal vs measured rate check is completed  
- At least one visualization per key modality is generated  
- Metadata summary JSON and quality CSV are exported  
- Warnings are explicit and reproducible  


## 1.1 Initialize the Data Provider

The VrsDataProvider is the core interface provided by projectaria_tools to interact with recorded data. Instead of manually parsing binary chunks, this provider abstracts the complexity of the .vrs container, allowing us to query specific sensor streams by index or timestamp. 

In this section, we will establish the connection to our local sample files and validate that the provider can correctly open the data stream.

In [ ]:
import os
import json
from projectaria_tools.core import data_provider

# Define paths to the .vrs and .json files
DATA_DIR = os.path.join("..", "data", "raw", "recording_test")
VRS_PATH = os.path.join(DATA_DIR, "recording_test.vrs")
JSON_PATH = os.path.join(DATA_DIR, "recording_test.json")

print("VRS exists:", os.path.exists(VRS_PATH), "|", VRS_PATH)
print("JSON exists:", os.path.exists(JSON_PATH), "|", JSON_PATH)


VRS exists: True | ..\data\raw\recording_test\recording_test.vrs
JSON exists: True | ..\data\raw\recording_test\recording_test.json


In [ ]:
# Load the .json metadata 
if os.path.exists(JSON_PATH):
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)
    print("JSON loaded correctly")
    print("Top-level keys:", list(metadata.keys()))
else:
    metadata = None
    print("JSON file not found")

JSON loaded correctly
Top-level keys: ['companion_version', 'custom_profile', 'data_quality_stats', 'device_id', 'device_test_automation_enabled', 'device_version', 'encryption_enabled', 'end_time', 'file_size', 'filename', 'firmware_version', 'is_vio_preprocessing_enabled', 'needs_anonymization', 'ntp_server_hostname', 'ntp_time_enabled', 'recording_profile', 'rgb_isp_tuning_version', 'start_time', 'telemetry_id', 'ticsync_enabled', 'timecode_enabled', 'timecode_trigger_enabled', 'type', 'vio_data_saving_enabled', 'vio_enabled', 'vio_setup_mode', 'campaign', 'cohort', 'hash_ca_id', 'name', 'tags']


In [ ]:
# Initialize the data provider for the .vrs file
if os.path.exists(VRS_PATH):
    provider = data_provider.create_vrs_data_provider(VRS_PATH) # 
    if provider:
        print("Data provider initialized successfully")
    else:
        print("Failed to initialize data provider")
else:
    provider = None
    print("VRS file not found")

Data provider initialized successfully


## 1.2 Stream Discovery and Canonical Table

In this section, we will inspect all streams exposed by the VRS provider and build a canonical table that summarizes their identity and basic properties. This will give a clear overview of the recording structure and will create a reliable reference for future data access and analysis steps.

In [12]:
import pandas as pd

# Assure the provider was created 
if "provider" not in globals() or provider is None:
    raise RuntimeError("ERROR: Provider is not initialized.")

# Helper function to assign a simple modality label from stream name/label
def infer_modality(label: str) -> str:
    """Infer a simple modality label from the stream label or name."""
    text = (label or "").lower()
    if any(k in text for k in ["camera", "image", "rgb", "slam"]):
        return "image"
    if any(k in text for k in ["imu", "accel", "gyro"]):
        return "imu"
    if "audio" in text:
        return "audio"
    if any(k in text for k in ["baro", "mag", "gps"]):
        return "other-sensor"
    return "unknown"

# Read all available stream IDs from the provider
stream_ids = provider.get_all_streams()
stream_ids 

[214-1, 247-1, 281-1, 282-1, 283-1, 1201-1, 1201-2, 1202-1, 1202-2, 1203-1]

In [15]:
# Iterate over all streams and collect metadata for the canonical table
rows = []
for sid in stream_ids:
    # Get the label and the sample count for each stream
    label = provider.get_label_from_stream_id(sid)
    sample_count = provider.get_num_data(sid)

    rows.append(
        {
            "stream_id": str(sid),
            "label": label,
            "modality": infer_modality(label),
            "sample_count": sample_count,
        }
    )

# Build canonical table
streams_df = pd.DataFrame(rows).sort_values(by=["modality", "stream_id"]).reset_index(drop=True)

# Show full table
print(f"Discovered streams: {len(streams_df)}")
display(streams_df)


Discovered streams: 10


,stream_id,label,modality,sample_count
0,1201-1,camera-slam-left,image,1185
1,1201-2,camera-slam-right,image,1185
2,214-1,camera-rgb,image,1185
3,1202-1,imu-right,imu,39454
4,1202-2,imu-left,imu,31845
5,1203-1,mag0,other-sensor,394
6,247-1,baro0,other-sensor,1966
7,281-1,gps,other-sensor,40
8,282-1,wps,unknown,69
9,283-1,bluetooth,unknown,0


In [16]:
# Quick modality summary
summary_df = streams_df.groupby("modality", dropna=False).size().reset_index(name="num_streams")
display(summary_df)

,modality,num_streams
0,image,3
1,imu,2
2,other-sensor,3
3,unknown,2


## 1.3 JSON Metadata Parsing

The recording JSON file contains critical metadata about the recording session, including device configuration, sensor specifications, and environmental context.
 
In this section, we will parse the JSON file and extract key fields that are relevant for understanding the conditions under which the data was recorded.

In [36]:
import pandas as pd

# Assure metadata is available
if "metadata" not in globals() or metadata is None:
    raise RuntimeError("ERROR: Metadata is not available.")

# Helper function to read nested values using dot notation
def get_by_path(data, path):
    current = data
    for key in path.split("."):
        if isinstance(current, dict) and key in current:
            current = current[key]
        else:
            return None
    return current

# Field-to-path mapping aligned with JSON schema
field_map = {
    "firmware_version": "firmware_version",
    "recording_profile": "recording_profile",
    "start_time": "start_time",
    "timecode_enabled": "timecode_enabled",
    "timecode_trigger_enabled": "timecode_trigger_enabled",
    "custom_profile_description": "custom_profile.description",
    "imu_1_enabled": "custom_profile.imu_1.enabled",
    "imu_1_rate_hz": "custom_profile.imu_1.data_rate_hz",
    "imu_2_enabled": "custom_profile.imu_2.enabled",
    "imu_2_rate_hz": "custom_profile.imu_2.data_rate_hz",
    "gps_enabled": "custom_profile.gps.enabled",
    "gps_rate_hz": "custom_profile.gps.data_rate_hz",
    "barometer_enabled": "custom_profile.barometer.enabled",
    "barometer_rate_hz": "custom_profile.barometer.data_rate_hz",
    "audio_enabled": "custom_profile.audio.enabled",
    "et_camera_enabled": "custom_profile.et_camera.enabled",
}

# Build output table with field, path, and extracted value
rows = []
for field, path in field_map.items():
    value = get_by_path(metadata, path)
    rows.append({"field": field, "path": path, "value": value})

field_map_df = pd.DataFrame(rows)

display(field_map_df)

,field,path,value
0,firmware_version,firmware_version,3.32.1
1,recording_profile,recording_profile,custom_profile_1222282970
2,start_time,start_time,1768918009
3,timecode_enabled,timecode_enabled,False
4,timecode_trigger_enabled,timecode_trigger_enabled,False
5,custom_profile_description,custom_profile.description,"Custom (RGB 30fps 2MP, SLAM 30fps VGA; IMUs, M..."
6,imu_1_enabled,custom_profile.imu_1.enabled,True
7,imu_1_rate_hz,custom_profile.imu_1.data_rate_hz,1000
8,imu_2_enabled,custom_profile.imu_2.enabled,True
9,imu_2_rate_hz,custom_profile.imu_2.data_rate_hz,800


In [37]:
# Show top-level structure overview for quick inspection
structure_rows = []
for key, value in metadata.items():
    value_type = type(value).__name__
    if isinstance(value, (str, int, float, bool)) or value is None:
        preview = value
    elif isinstance(value, list):
        preview = f"<list, len={len(value)}>"
    elif isinstance(value, dict):
        preview = f"<dict, keys={len(value.keys())}>"
    else:
        preview = f"<{value_type}>"

    structure_rows.append({"top_level_key": key, "type": value_type, "preview": preview})

structure_df = pd.DataFrame(structure_rows).sort_values("top_level_key").reset_index(drop=True)
display(structure_df)

,top_level_key,type,preview
0,campaign,NoneType,None
1,cohort,str,external_academic
2,companion_version,str,
3,custom_profile,dict,"<dict, keys=14>"
4,data_quality_stats,dict,"<dict, keys=10>"
5,device_id,str,36186c9d-a825-40e1-b517-5cbc5418ad31
6,device_test_automation_enabled,bool,False
7,device_version,str,49772820000000070
8,encryption_enabled,bool,False
9,end_time,int,1768918049
